In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('Dataset .csv')

In [4]:
df.head(5)

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


In [5]:
df.isnull().sum()

Restaurant ID           0
Restaurant Name         0
Country Code            0
City                    0
Address                 0
Locality                0
Locality Verbose        0
Longitude               0
Latitude                0
Cuisines                9
Average Cost for two    0
Currency                0
Has Table booking       0
Has Online delivery     0
Is delivering now       0
Switch to order menu    0
Price range             0
Aggregate rating        0
Rating color            0
Rating text             0
Votes                   0
dtype: int64

In [8]:
df_clean = df.dropna(subset = ['Cuisines']).copy()

In [9]:
df.isnull().sum()

Restaurant ID           0
Restaurant Name         0
Country Code            0
City                    0
Address                 0
Locality                0
Locality Verbose        0
Longitude               0
Latitude                0
Cuisines                0
Average Cost for two    0
Currency                0
Has Table booking       0
Has Online delivery     0
Is delivering now       0
Switch to order menu    0
Price range             0
Aggregate rating        0
Rating color            0
Rating text             0
Votes                   0
dtype: int64

In [10]:
df_clean['Cuisines_list'] = df_clean['Cuisines'].apply(lambda x: [c.strip() for c in str(x).split(',')])

In [14]:
from sklearn.preprocessing import  MultiLabelBinarizer
mlb = MultiLabelBinarizer()  #using  MultiLabelBinarizer instead of Onehotencoder due to every column having multiple labels
cuisine_encoded = pd.DataFrame(
    mlb.fit_transform(df_clean['Cuisines_list']), 
    columns=mlb.classes_, 
    index=df_clean.index
)

In [15]:
df_processed = pd.concat([df_clean, cuisine_encoded], axis=1)

In [16]:
df_processed.head(5)

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Tex-Mex,Thai,Tibetan,Turkish,Turkish Pizza,Unknown,Vegetarian,Vietnamese,Western,World Cuisine
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,0,0,0,0,0,0,0,0,0,0
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,0,0,0,0,0,0,0,0,0,0
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,0,0,0,0,0,0,0,0,0,0
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,0,0,0,0,0,0,0,0,0,0
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,0,0,0,0,0,0,0,0,0,0


In [17]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

def recommend_restaurants(df_processed, mlb_classes, user_cuisines, max_price, min_rating, top_n=5):
    
    filtered_df = df_processed[
        (df_processed['Price range'] <= max_price) & 
        (df_processed['Aggregate rating'] >= min_rating)
    ].copy()
    
    if filtered_df.empty:
        return "No restaurants found matching your price and rating criteria."
    user_vector = pd.DataFrame(np.zeros((1, len(mlb_classes))), columns=mlb_classes)
    valid_cuisines = [c for c in user_cuisines if c in mlb_classes]
    if not valid_cuisines:
        return filtered_df.sort_values(by='Aggregate rating', ascending=False).head(top_n)
    user_vector[valid_cuisines] = 1
    restaurant_cuisine_matrix = filtered_df[mlb_classes]
    similarity_scores = cosine_similarity(user_vector, restaurant_cuisine_matrix)
    filtered_df['Similarity Score'] = similarity_scores[0]
    recommended = filtered_df.sort_values(
        by=['Similarity Score', 'Aggregate rating'], 
        ascending=[False, False]
    )
    
    return recommended.head(top_n)[['Restaurant Name', 'Cuisines', 'Price range', 'Aggregate rating', 'Similarity Score']]